In [ ]:
%%javascript
IPython.OutputArea.auto_scroll_threshold = 9999;

In [7]:
from IPython.core.interactiveshell import InteractiveShell  
InteractiveShell.ast_node_interactivity = "all"
from IPython.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>")) 

In [28]:
from dash import Dash, html, dcc, Input, Output, dash_table, callback, State, ClientsideFunction
from dash.exceptions import PreventUpdate
import dash_bootstrap_components as dbc
import collections
import pandas as pd
import wget
import json
import random
from application.dash.sidebar import spes_options, uga_options, ciblage_options 
from dash_extensions.javascript import assign, arrow_function, Namespace
import dash_leaflet as dl
import dash_leaflet.express as dlx
import re
from datetime import datetime
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import relationship, Session
from flask_sqlalchemy import SQLAlchemy
from sqlalchemy import Column, Integer, Sequence, String, Text, DateTime, ForeignKey, create_engine
import datetime
from flask import Flask, request, url_for, render_template, render_template_string
from datatables import DataTable
import io

from app import app
app.app_context().push()

def json_js_to_geojson(json_js_file):
    import json

    with open(f'assets/js/{json_js_file}.js') as dataFile:
        data = dataFile.read()

        obj = data[data.find('{'): data.rfind('}') + 1]

        geojson = json.loads(obj)

    with open(f'assets/{json_js_file}.json', 'w') as j:
        j.write(json.dumps(geojson).replace(": ", ":").replace(", ", ","))

    return geojson


def p_popup(pharmacy):
    return f"""
            <b>{pharmacy['nom']}</b>
            <br>{pharmacy["adr"]}
            <br>{pharmacy["cp"]} {pharmacy["ville"]} ({pharmacy["uga"]})
            <br>{pharmacy["tel"]}
            
        """


def c_popup(row):
    return render_template("partials/front.html", row=row)


def data_to_geojson(data_df):
    df_geojson = dlx.dicts_to_geojson(
        [{**c, **dict(popup=c_popup(c))} if c['spe'] != "" else {**c, **dict(popup=p_popup(c))} for c in
         data_df.to_dict('records')])
    with open(f'assets/df_geojson.json', 'w') as j:
        j.write(json.dumps(df_geojson).replace(": ", ":").replace(", ", ","))
    return df_geojson

from application.dash.biocodex.functions import build_modal, build_tile_front, doctor_colors
from application.dash.biocodex.functions import prepare_data, join_id_adr_cdb
from application.dash.biocodex.functions import mean_lat, mean_lon, datatable_cols, styles, legend1, legend2

In [9]:
bermuda = dlx.dicts_to_geojson([dict(lat=32.299507, lon=-64.790337, popup="Bermuda")])

In [10]:
bermuda

{'type': 'FeatureCollection',
 'features': [{'type': 'Feature',
   'geometry': {'type': 'Point', 'coordinates': [-64.790337, 32.299507]},
   'properties': {'popup': 'Bermuda'}}]}

In [ ]:
from application.config import basedir
from dotenv import load_dotenv
import os

load_dotenv(os.path.join(basedir, '.env'))
DATABASE_URL = os.environ.get('DATABASE_URL').replace('postgres://', 'postgresql://')
engine = create_engine(DATABASE_URL)

In [14]:
df = join_id_adr_cdb()

In [15]:
data_df = prepare_data(df)

In [18]:
df_geojson = data_to_geojson(data_df)

In [20]:
ns = Namespace('dashExtensions','default')

ugas = ["75AUT", "75PAS", "75TRO", "75ELY", "75INV", "75GRE", "75VAU", "75MNP", "75PER", "75TER", "92LEV", "92NEU"]

with open("assets/uga_gpd.json", "r") as uga_json:
    uga_geojson = json.loads(uga_json.read())

sector_features = [
    uga_geojson["features"][i] for i in range(len(uga_geojson["features"])) if uga_geojson["features"][i]["properties"]["CODE_UGA"] in ugas
]
uga_geojson["features"] = sector_features


ugas_layer = dl.GeoJSON(
    data=uga_geojson,
    hideout=dict(selected=["75AUT"]),
    filter=ns('uga_geojson_filter'),
    style=ns('uga_style_handle'),
    hoverStyle=arrow_function(dict(weight=3, color='red', dashArray='')),
    zoomToBounds=True,
    id="uga-geojson",
)

data_layer = dl.GeoJSON(
    data=df_geojson,
    hideout=dict(ugas_selected=["75AUT"], spes_selected=["GY"]),
    filter=ns('geojson_filter'),
    zoomToBounds=True,
    pointToLayer=ns('cible_icon'),
    id="data-geojson"
)

In [22]:
external_scripts = [
    'https://code.jquery.com/jquery-3.3.1.min.js',
    'https://cdn.datatables.net/v/dt/dt-1.10.18/datatables.min.js',
    "https://cdn.datatables.net/1.13.6/js/dataTables.bootstrap5.min.js",
    {"type": "text/javascript", "src": "application/dash/assets/js/flip.js"},
    {"type": "text/javascript", "src": "application/dash/assets/js/modalize.js"},
    {"type": "text/javascript", "src": "js/leaflet.extra-markers.min.js"},
]
external_stylesheets = [
    dbc.themes.BOOTSTRAP, 
    "https://use.fontawesome.com/releases/v5.0.3/css/all.css",
    {"rel":"stylesheet", "type":"text/css", "src":'https://cdn.datatables.net/v/dt/dt-1.10.18/datatables.min.css'},
    {"rel": "stylesheet", "type": "text/css", "src": "css/leaflet.extra-markers.min.css"},
    {"rel": "stylesheet", "type": "text/css", "src": "css/sidebar.css"},
    {"rel": "stylesheet", "type": "text/css", "src": "css/hamburgers.css"},
    {"rel": "stylesheet", "type":"text/css", "src":'https://cdnjs.cloudflare.com/ajax/libs/font-awesome/5.14.0/css/all.min.css'},
    {"rel": "stylesheet", "type":"text/css", "src":'https://fonts.googleapis.com/css?family=Open+Sans'},
    {"rel": 'stylesheet', "type":'text/css', "src":'https://use.fontawesome.com/releases/v5.15.1/css/all.css'}
]

app = Dash(__name__)

app.layout = html.Div(
        [
            dl.Map(
                [
                   dl.LayersControl(
                        [
                            dl.BaseLayer(dl.TileLayer(), name="base", checked=True),
                            dl.Overlay(dl.LayerGroup(ugas_layer, id="ugas_layer_group"), name="ugas", checked=True),
                            dl.Overlay(dl.LayerGroup(data_layer, id="data_layer_group"), name="data", checked=True),
                            #dl.Overlay(dl.LayerGroup(target_layer, id="target_layer_group"), name="ciblés", checked=True),
                            #dl.Overlay(dl.LayerGroup(untarget_layer, id="untarget_layer_group"), name="non ciblés", checked=False),
                            #info
                        ]
                   ),
                    dl.FullScreenControl(),
                    dl.LocateControl(locateOptions={'enableHighAccuracy': True})
               ],
               center=(mean_lat, mean_lon),
               zoom=11,
               style={'height': '90vh', 'position': 'absolute', 'width': '90%'}
            )
        ],
        id='map'
    )


app.run_server(debug=True)

In [ ]:
ugas = ["75AUT", "75PAS", "75TRO", "75INV", "75ELY", "75GRE", "75VAU", "75MNP", "75PER", "75TER", "92LEV", "92NEU"]
spes = ["GY", "MG-GY", "SF", "MG", "GE", "PE", "PE-PSY", "PSY", "NE", "PHA"]

In [32]:
import os

In [35]:
from app import app
app.app_context().push()

In [33]:
from application.config import basedir
from dotenv import load_dotenv

load_dotenv(os.path.join(basedir, '.env'))
DATABASE_URL = os.environ.get('DATABASE_URL').replace('postgres://', 'postgresql://')

# an Engine, which the Session will use for connection
# resources
engine = create_engine(DATABASE_URL)

True

In [37]:
from application.dash.biocodex.models import Connections, Identity, Adress, Cdb

In [38]:
def join_id_adr_cdb():

    with Session(engine) as session:
        id_adr_cdb = pd.read_sql_query(
            sql=session.query(
                Connections.doc_id, Identity.nom, Identity.prenom, Identity.spe, Identity.pot, Identity.pvm,
                Identity.nv22, Identity.cib, Identity.dec,
                Adress.eta, Adress.uga, Adress.adr, Adress.cp, Adress.ville, Adress.tel, Adress.lat, Adress.lon,
                Cdb.mode, Cdb.com, extract("EPOCH", Cdb.ddv), extract("EPOCH", Cdb.dpv), Cdb.rdv, Cdb.rec, Cdb.pk,
                Cdb.lun_mat, Cdb.lun_am, Cdb.mar_mat, Cdb.mar_am, Cdb.mer_mat, Cdb.mer_am, Cdb.jeu_mat, Cdb.jeu_am,
                Cdb.ven_mat, Cdb.ven_am
            ).join(Identity).join(Adress).join(Cdb).statement,
            con=engine
        )
        session.close()
        id_adr_cdb.columns = [
            'id', 'nom', 'prenom', 'spe', 'pot', 'pvm', 
            'nv22', 'cib', 'dec',
            'eta', 'uga', 'adr', 'cp', 'ville', 'tel', 'lat', 'lon', 
            'mode', 'com', 'ddv', 'dpv', 'rdv', 'rec', 'pk',
            'lun_mat', 'lun_am', 'mar_mat', 'mar_am', 'mer_mat', 'mer_am', 'jeu_mat', 'jeu_am',
            'ven_mat', 'ven_am'
        ]

        df=prepare_data(id_adr_cdb)

    return df
df = join_id_adr_cdb()

NameError: name 'extract' is not defined

In [24]:
df["cib"].unique()

array([ 0,  3,  2,  4, -1])

In [25]:
data_df = prepare_data(df)

In [26]:
data_df["cib"].unique()

array(['HC', 3, 2, 4, -1], dtype=object)

    for col in ['nv2022', 'ciblage']:
        df[col] = np.where(df[col]==0, '', df[col])

    df_geojson = dlx.dicts_to_geojson([{**c, **dict(popup=c_popup(c))} for c in df.to_dict('records')])

    with open(f'assets/df_geojson.json', 'w') as j:
        j.write(json.dumps(df_geojson).replace(": ", ":").replace(", ", ","))

    target_geojson = dlx.dicts_to_geojson([{**c, **dict(popup=c_popup(c))} for c in df[df['ciblage']!=''].to_dict('records')])

    with open(f'assets/target_geojson.json', 'w') as j:
        j.write(json.dumps(target_geojson).replace(": ", ":").replace(", ", ","))

    with open(f'app/dashboards/assets/target_geojson.json', 'w') as j:
        j.write(json.dumps(target_geojson).replace(": ", ":").replace(", ", ","))

    untarget_geojson = dlx.dicts_to_geojson([{**c, **dict(popup=c_popup(c))} for c in df[df['ciblage']==''].to_dict('records')])

    with open(f'assets/untarget_geojson.json', 'w') as j:
        j.write(json.dumps(untarget_geojson).replace(": ", ":").replace(", ", ","))

    with open(f'app/dashboards/assets/untarget_geojson.json', 'w') as j:
        j.write(json.dumps(untarget_geojson).replace(": ", ":").replace(", ", ","))

In [ ]:
with open("assets/uga_gpd.json", "r") as uga_json:
    uga_geojson = json.loads(uga_json.read())

sector_features = [uga_geojson["features"][i] for i in range(len(uga_geojson["features"])) if uga_geojson["features"][i]["properties"]["CODE_UGA"] in ugas]
uga_geojson["features"] = sector_features


In [ ]:
ns = Namespace('dashExtensions','default')

ugas_layer = dl.GeoJSON(
    data=uga_geojson,
    hideout=dict(selected=[]),
    filter=ns('uga_geojson_filter'),
    style=ns('uga_style_handle'),
    hoverStyle=arrow_function(dict(weight=3, color='red', dashArray='')),
    zoomToBounds=True,
    id="uga-geojson",
)




In [ ]:
external_scripts=["js/leaflet.extra-markers.min.js"]
external_stylesheets=[dbc.themes.BOOTSTRAP, dbc.icons.FONT_AWESOME, "css/leaflet.extra-markers.min.css"]

app = Dash(external_scripts=external_scripts, external_stylesheets=external_stylesheets)

app.layout = html.Div(
    [
        dcc.Checklist(id="uga-cl", value=["75AUT"], options=[{"value": uga, "label": uga} for uga in ugas], inline=True),
        dcc.Checklist(id="spe-cl", value=["GY"], options=[{"value": spe, "label": spe} for spe in df["spe"].unique()], inline=True),
        dl.Map(
            [
               dl.LayersControl(
                    [
                        dl.BaseLayer(dl.TileLayer(), name="base", checked=True),
                        dl.Overlay(dl.LayerGroup(ugas_layer, id="ugas_layer_group"), name="ugas", checked=True),
                        dl.Overlay(dl.LayerGroup(pharmas_layer, id="pharmas_layer_group"), name="pharmas", checked=True),
                        dl.Overlay(dl.LayerGroup(target_layer, id="target_layer_group"), name="ciblés", checked=True),
                        dl.Overlay(dl.LayerGroup(untarget_layer, id="untarget_layer_group"), name="non ciblés", checked=False),
                        info
                    ]
               ),
                dl.FullScreenControl(),
                dl.LocateControl(locateOptions={'enableHighAccuracy': True})               
           ],
           center=(mean_lat, mean_lon), 
           zoom=12, 
           style={'height': '90vh'}
        )
    ]
)

@app.callback(
    Output("uga-geojson", "hideout"),
    Input("uga-cl", "value"),
    State("uga-geojson", "hideout"),
)
def toggle_select(ugas_selected, uga_hideout):
    uga_hideout["selected"] = ugas_selected
    return uga_hideout

@app.callback(
    Output("pharma-geojson", "hideout"),
    Input("uga-cl", "value"),
    State("pharma-geojson", "hideout")
)
def toggle_select(ugas_selected, pharma_hideout):
    pharma_hideout["ugas_selected"] = ugas_selected
    return pharma_hideout

@app.callback(
    Output("target-geojson", "hideout"),
    Input("uga-cl", "value"),
    Input("spe-cl", "value"),
    State("target-geojson", "hideout")
)
def toggle_select(ugas_selected, spes_selected, target_hideout):
    target_hideout["ugas_selected"] = ugas_selected
    target_hideout["spes_selected"] = spes_selected
    return target_hideout

@app.callback(
    Output("untarget-geojson", "hideout"),
    Input("uga-cl", "value"),
    Input("spe-cl", "value"),
    State("untarget-geojson", "hideout")
)
def toggle_select(ugas_selected, spes_selected, untarget_hideout):
    untarget_hideout["ugas_selected"] = ugas_selected
    untarget_hideout["spes_selected"] = spes_selected
    return untarget_hideout

@app.callback(
    Output("info", "children"), 
    Input("uga-geojson", "hoverData")
)
def info_hover(feature):
    return get_info(feature)

    
app.run_server(debug=True)

In [ ]:
app = Dash(name="maps_app", , "css/geojson.css"])

app.layout = html.Div(
    [
        dcc.Checklist(id="uga-cl", value=uga_cl_default, options=uga_cl_opt, inline=True, labelStyle={"display": "inline-block"}),        
        
        dbc.Button("Show UGAs", id="uga-btn"),
        dbc.Button("Show pharmacies", id="pharma-btn"),
        dbc.Button("Show doctors", id="doc-btn"),
        dl.Map(
            [
                dl.LayersControl(
                    [
                        dl.BaseLayer(dl.TileLayer(), name="base", checked=True),
                        dl.Overlay(
                            dl.LayerGroup(
                                ugas_layer,
                                id="ugas_layer_group"
                            ),
                            name="ugas",
                            checked=True
                        ),
                        dl.Overlay(
                            dl.LayerGroup(
                                pharmas_layer,
                                id="pharmas_layer_group"
                            ),
                            name="pharmacies",
                            checked=True
                        ),
                        dl.Overlay(
                            dl.LayerGroup(
                                docs_layer,
                                id="docs_layer_group"
                            ),
                            name="doctors",
                            checked=True
                        )
                    ]
                ),
                dl.FullScreenControl(),
                dl.LocateControl(locateOptions={'enableHighAccuracy': True}),
                info
            ],
            center=(ciblage["lat"].mean(), ciblage["lon"].mean()), 
            zoom=11, 
            style={'height': '90vh'}
        )
    ]
)




app.clientside_callback(
    "function(x){return x;}", 
    Output("uga_geojson", "hideout"), 
    Input("uga-cl", "value")
)





    @app.callback(Output("ugas_layer_group", "children"), Input("uga-btn", "n_clicks"))
    def gen_uga_layer(_):
        return ugas_layer

    @app.callback(Output("pharmas_layer_group", "children"), Input("pharma-btn", "n_clicks"))
    def gen_uga_layer(_):
        return pharmas_layer

    @app.callback(Output("docs_layer_group", "children"), Input("doc-btn", "n_clicks"))
    def gen_uga_layer(_):
        return docs_layer
    
    app.clientside_callback(
    "function(x){return x;}", 
    Output("doc_geojson", "hideout"), 
    Input("uga-cl", "value")
    )
    
    app.clientside_callback(
        "function(x){return x;}", 
        Output("pharma_geojson", "hideout"), 
        Input("uga-cl", "value")
    )

In [ ]:
app.run_server(debug=True,dev_tools_hot_reload=True)

m = Map(center=(ciblage["lat"].mean(), ciblage["lon"].mean()), zoom=12)


for uga in ugas:
    layers=[]
    uga_data = {
        'type': data['type'],
        'crs': data['crs'],
        'features': [data['features'][i] for i in range(len(data['features'])) if data['features'][i]['properties']['CODE_UGA'] == uga]
    }
    geo_json = GeoJSON(
        data=uga_data,
        style={
            'opacity': 1, 'dashArray': '9', 'fillOpacity': 0.1, 'weight': 1
        },
        hover_style={
            'color': 'white', 'dashArray': '0', 'fillOpacity': 0.5
        },
        style_callback=random_color,
        zoomToBounds=True
    )
    layers.append(geo_json)
    pharma_markers=[]
    for i, pharmacy in pharma[pharma["uga"]==uga].iterrows():

        popup = HTML()
        popup.value = f"""
            <b>{pharmacy['nom']}</b>
            <br>{pharmacy["adresse"]}
            <br>{pharmacy["cp"]} {pharmacy["ville"]} ({pharmacy["uga"]})
            <br>{pharmacy["tel"]}
            <br>CA UL (CMA juin 23): {pharmacy["ca ul cma juin 23"]} (rang {pharmacy["rang ca uk"]})
            <br>Ciblage DP:{pharmacy["ciblage dp"]} DSO:{pharmacy["ciblage dso"]} DM:{pharmacy["ciblage dm"]}                
        """

        marker = Marker(
            icon = AwesomeIcon(
                name='plus-square',
                marker_color='green',
                icon_color='white',
                spin=False
            ), 
            location=(pharmacy["lat"], pharmacy["lon"]),
            popup=popup

        )    

        pharma_markers.append(marker)
        pharma_cluster=MarkerCluster(markers=(pharma_markers), name="pharma")
    layers.append(pharma_cluster)
        
    for spe in spe_to_color.keys():
        cible_spe=ciblage[(ciblage["spe"]==spe)&(ciblage["uga"]==uga)].copy()
        spe_markers=[]
        for i, doc in cible_spe.iterrows():
            marker = Marker(
                icon=AwesomeIcon(
                    name='user-md',
                    marker_color=spe_to_color[doc["spe"]]
                ), 
                location=(doc["lat"], doc["lon"])
            )
            spe_markers.append(marker)
        spe_cluster=MarkerCluster(markers=(spe_markers), name=spe)
        layers.append(spe_cluster)
    
    layer_group=LayerGroup(layers=(layers), name=uga)

    m.add_layer(layer_group)


m.add_control(LayersControl(position='topright'))
m.add_control(FullScreenControl())
m.add_control(ScaleControl(position='bottomleft'))

marker = Marker(icon=AwesomeIcon(name="check", marker_color='green', icon_color='darkred'))

m.add_control(SearchControl(
  position="topleft",
  layer=layer_group,
  zoom=15,
  marker=marker
))

display(m)

    import random

    number_of_colors = len(ugas)
    colors=[]

    color = ["#"+''.join([random.choice('0123456789ABCDEF') for j in range(6)]) for i in range(number_of_colors)]
    colors.append(color)
    colors=colors[0]

    uga_colors = {}
    for u, c in zip(ugas, colors):
        uga_colors[u]=c